# Leg-2 encoder pilot

Relabel a lane with a different dense encoder and diff against the shipped labels. **Only the dense leg changes** — sparse stays `Qdrant/bm25`, so it is a fixed control: every movement is dense's fault.

Every query falls in exactly one of three buckets, decided by its three route scores:

| bucket | meaning |
| --- | --- |
| **`all_zero`** | all three scores are 0 — nothing relevant found |
| **`all_tied`** | all three scores are the same — every route did equally well |
| **`routes_differ`** | anything else — the routes disagree |

That is `labels.outcome_shape`, the repo's own definition. One section per bucket: **where did those rows go, and what moved in the scores.**

In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import os, sys, pathlib

SRC = pathlib.Path.cwd()
SRC = SRC if (SRC / 'scripts').exists() else SRC / 'src'
sys.path.insert(0, str(SRC))

import pandas as pd
from dotenv import load_dotenv
from qdrant_client import QdrantClient

from composition.pool_v3 import SCORES
from scripts.legb import LEGB_DIR, LegBPilot, e5_dense_cfg, qwen_dense_cfg

load_dotenv()
client = QdrantClient(
    url=os.environ['QDRANT_CLOUD_URL'],
    api_key=os.environ['QDRANT_CLOUD_API_KEY'],
    timeout=120,
    cloud_inference=True,
)
pd.set_option('display.width', 170)

## 1. What to draw

One entry per bucket, all labelled in a single pass below. Comment one out to skip it — its section then says `not sampled yet`.

`n` is **per lane**, fixed seed.

In [14]:
ENCODER = qwen_dense_cfg   # or e5_dense_cfg (local, free, no API key)
LANE    = 'crumb-legal-qa' # None = every pilot lane

DRAWS = {
    'all_tied':      ('all_tied', 60),
    'all_zero':      ('all_zero', 100),
    'routes_differ': ('routes_differ', 100),
}

pilots = {name: LegBPilot(client, ENCODER(), sample=draw)
          for name, draw in DRAWS.items()}

pd.concat([p.plan().assign(draw=name) for name, p in pilots.items()],
          ignore_index=True)[['draw', 'lane', 'to_label', 'indexed']]

[antique] sample asked 60 all_tied rows, drew 0 — that is the whole available population
[antique] sample asked 100 all_zero rows, drew 0 — that is the whole available population
[antique] sample asked 100 routes_differ rows, drew 0 — that is the whole available population


,draw,lane,to_label,indexed
0,all_tied,scirgen-geo-en,60,False
1,all_tied,crumb-legal-qa,60,True
2,all_tied,antique,0,False
3,all_zero,scirgen-geo-en,100,False
4,all_zero,crumb-legal-qa,100,True
5,all_zero,antique,0,False
6,routes_differ,scirgen-geo-en,100,False
7,routes_differ,crumb-legal-qa,100,True
8,routes_differ,antique,0,False


## 2. Label

Idempotent — a full collection skips indexing, an already-labelled query is not rescored. This is the paid step.

In [15]:
for name, p in pilots.items():
    print(f'\n########## {name} ##########')
    failed = p.index_and_label(lane=LANE)
    if failed:
        print('  FAILED:', failed)


########## all_tied ##########
=== [1/1] crumb-legal-qa (openrouter/qwen/qwen3-embedding-8b, max_workers=8) ===


label:crumb-legal-qa: 100%|██████████| 1/1 [00:30<00:00, 30.33s/chunk, differ=10, labelled=26, tied=16, zero=0]


[crumb-legal-qa] +26 leg-2 labels  differ 10 | tied 16 | zero 0

########## all_zero ##########
=== [1/1] crumb-legal-qa (openrouter/qwen/qwen3-embedding-8b, max_workers=8) ===
[crumb-legal-qa] +0 leg-2 labels  differ 0 | tied 0 | zero 0

########## routes_differ ##########
=== [1/1] crumb-legal-qa (openrouter/qwen/qwen3-embedding-8b, max_workers=8) ===


label:crumb-legal-qa: 100%|██████████| 1/1 [01:25<00:00, 85.45s/chunk, differ=82, labelled=91, tied=3, zero=6]

[crumb-legal-qa] +91 leg-2 labels  differ 82 | tied 3 | zero 6


## 3. Pair the legs

`paired` = one row per query, both legs' three scores side by side, plus each leg's bucket and winning route.

First the control: **`sparse` must be identical on every row.** It is the leg we did not touch. If it moved, this is not a controlled comparison and nothing below counts.

In [16]:
SHORT = {'score_dense_only': 'dense', 'score_sparse_only': 'sparse',
         'score_pure_rrf': 'rrf'}

pilot = next(iter(pilots.values()))
leg2 = pd.read_parquet(LEGB_DIR / 'labels.parquet')[['dataset', 'query_id'] + SCORES]
leg1 = pilot._pool.labels()[['dataset', 'query_id'] + SCORES]
paired = leg1.merge(leg2, on=['dataset', 'query_id'], suffixes=('_1', '_2'))

for leg in ('1', '2'):
    cols = [c + '_' + leg for c in SCORES]
    paired['max_' + leg] = paired[cols].max(axis=1)
    paired['min_' + leg] = paired[cols].min(axis=1)
    # the repo's own three-way split: all zero / all the same / they differ
    paired['bucket_' + leg] = 'routes_differ'
    paired.loc[paired['max_' + leg] - paired['min_' + leg] <= 1e-9,
               'bucket_' + leg] = 'all_tied'
    paired.loc[paired['max_' + leg] <= 1e-9, 'bucket_' + leg] = 'all_zero'
    paired['winner_' + leg] = paired[cols].idxmax(axis=1).map(
        lambda c: SHORT[c.rsplit('_', 1)[0]])

for col, short in SHORT.items():
    changed = int((paired[col + '_1'].round(9) != paired[col + '_2'].round(9)).sum())
    print(f'{short:7} changed on {changed:>4}/{len(paired)}'
          + ('   <-- MUST BE 0 (the fixed control)' if short == 'sparse' else ''))

print(f'\n{len(paired)} rows paired')
paired['bucket_1'].value_counts().rename('drawn from')

dense   changed on  317/498
sparse  changed on    0/498   <-- MUST BE 0 (the fixed control)
rrf     changed on  295/498

498 rows paired


bucket_1
routes_differ    291
all_tied         107
all_zero         100
Name: drawn from, dtype: int64

### Where everything went

The whole result in one table: leg-1 bucket down the side, leg-2 bucket across. The diagonal stayed put.

In [ ]:
pd.crosstab(paired['bucket_1'], paired['bucket_2'], margins=True)

### The one number: did retrieval find the judged document?

Ignore labels and margins for a second. Each query has a judged relevant document. Either some route retrieved it (score > 0) or none did (all zero). That is the encoder's actual job, and it is the cleanest comparison available:

- **restored** — leg-1 found nothing, leg-2 found it
- **destroyed** — leg-1 found it, leg-2 lost it

Everything downstream (which route wins, whether the margin certifies) is labelling *policy*. This is the encoder.

In [ ]:
restored = paired[(paired.max_1 <= 1e-9) & (paired.max_2 > 1e-9)]
destroyed = paired[(paired.max_1 > 1e-9) & (paired.max_2 <= 1e-9)]
print(f'restored  (nothing -> found) : {len(restored)}')
print(f'destroyed (found -> nothing) : {len(destroyed)}')
print(f'NET on information           : {len(restored) - len(destroyed):+d}')

dd = paired.score_dense_only_2 - paired.score_dense_only_1
print(f'\ndense moved down on {int((dd < -1e-9).sum())} rows, of which:')
print(f'  substantial (dropped > 0.5) : {int((dd < -0.5).sum())}')
print(f'  cosmetic    (dropped < 0.1) : {int((dd > -0.1).sum() - (dd.abs() <= 1e-9).sum())}')

---
# Eye test — real queries, per case

Six cases. `cases()` lists them with counts; `eye(name, n)` prints the queries with both legs' scores so you can judge whether the movement makes sense.

Read the score triples as `dense / sparse / rrf`. **`sparse` is identical across legs by construction** — it is the untouched control, so any change you see is dense.

In [ ]:
import textwrap

# query text lives in the leg-2 labels file
_q = pd.read_parquet(LEGB_DIR / 'labels.parquet')[['dataset', 'query_id', 'query']]
eyes = paired.merge(_q, on=['dataset', 'query_id'], how='left')
eyes['margin'] = eyes.max_2 - eyes[[c + '_2' for c in SCORES]].apply(
    lambda r: sorted(r)[-2], axis=1)
eyes['dd'] = eyes.score_dense_only_2 - eyes.score_dense_only_1

_tied = eyes.bucket_1 == 'all_tied'
_zero = eyes.bucket_1 == 'all_zero'
_broke = _tied & (eyes.bucket_2 == 'routes_differ')

CASES = {
    'zero_to_decisive':  (_zero & (eyes.bucket_2 == 'routes_differ') & (eyes.margin >= 0.4),
                          'all_zero -> found it, decisively (margin >= 0.4)'),
    'zero_to_nothing':   (_zero & (eyes.bucket_2 == 'all_zero'),
                          'all_zero -> still nothing (no encoder fixes these)'),
    'tied_to_decisive':  (_broke & (eyes.margin >= 0.4),
                          'all_tied -> broke decisively (margin >= 0.4)'),
    'tied_to_thin':      (_broke & (eyes.margin < 0.1) & (eyes.dd > 1e-9),
                          'all_tied -> broke thin, dense UP (0 < margin < 0.1)'),
    'tied_backwards':    (_broke & (eyes.dd < -1e-9),
                          'all_tied -> broke because dense went DOWN'),
    'destroyed':         ((eyes.max_1 > 1e-9) & (eyes.max_2 <= 1e-9),
                          'had a hit -> lost it entirely'),
}


def cases():
    return pd.DataFrame([{'case': k, 'rows': int(m.sum()), 'what': d}
                         for k, (m, d) in CASES.items()]).set_index('case')


def eye(case, n=5, sort='margin'):
    """Queries for one case, with both legs' scores as dense/sparse/rrf.

    Prints the query in FULL, wrapped. Several lanes are templated --
    crumb-legal-qa carries a 114-char boilerplate preamble ("Secondary methods
    of service are defined as...") -- so any head truncation cuts before the
    words that distinguish the rows and makes four different queries look like
    one repeated row. Nothing is hidden here for that reason.
    """
    mask, desc = CASES[case]
    rows = eyes[mask].sort_values(sort, ascending=False).head(n)
    print(f'{case} — {desc}\n{int(mask.sum())} rows, showing {len(rows)}\n' + '=' * 96)
    if rows.empty:
        print('  (none)')
        return None
    for _, r in rows.iterrows():
        print(textwrap.fill(str(r['query']), width=94,
                            initial_indent=f'[{r.query_id}] ', subsequent_indent=' ' * 8))
        print(f"   leg-1   {r.score_dense_only_1:.3f} / {r.score_sparse_only_1:.3f} / {r.score_pure_rrf_1:.3f}")
        print(f"   leg-2   {r.score_dense_only_2:.3f} / {r.score_sparse_only_2:.3f} / {r.score_pure_rrf_2:.3f}"
              f"   -> {r.winner_2}, margin {r.margin:.3f}, dense {r.dd:+.3f}\n")
    return None


cases()

### `zero_to_decisive`

The clean wins. Dense goes 0 -> 1.0: Qwen puts the judged doc at rank 1 where bge-small had nothing. Sparse stays 0, so BM25 never had these either.

In [26]:
eye('zero_to_decisive', n=5)

zero_to_decisive — all_zero -> found it, decisively (margin >= 0.4)
6 rows, showing 5
[7193] In an eviction action, can a tenant rebut/raise the defense that eviction is [...]
   leg-1   0.000 / 0.000 / 0.000
   leg-2   1.000 / 0.000 / 0.116   -> dense, margin 0.884, dense +1.000

[4350] Are eviction cases first heard in justice of the peace court? In the state of New Mexico
   leg-1   0.000 / 0.000 / 0.000
   leg-2   1.000 / 0.000 / 0.189   -> dense, margin 0.811, dense +1.000

[5697] Secondary methods of service are defined as those methods that may be used if the [...]
   leg-1   0.000 / 0.000 / 0.000
   leg-2   1.000 / 0.000 / 0.189   -> dense, margin 0.811, dense +1.000

[3441] For tenants residing in properties that go into foreclosure, does the law require [...]
   leg-1   0.000 / 0.000 / 0.000
   leg-2   1.000 / 0.000 / 0.189   -> dense, margin 0.811, dense +1.000

[38] Is there a state/territory law regulating residential evictions? In the state of Ohio
   leg-1   0.000 / 0.00

### `zero_to_nothing`

Both encoders fail identically. Look at the phrasing - these are near-duplicate templated queries. No dense model fixes this; it is what leg-3 (the LLM judge) exists for.

In [27]:
eye('zero_to_nothing', n=5)

zero_to_nothing — all_zero -> still nothing (no encoder fixes these)
62 rows, showing 5
[7957] Is the term 'Writ of eviction' used to refer to the order from the court to the [...]
   leg-1   0.000 / 0.000 / 0.000
   leg-2   0.000 / 0.000 / 0.000   -> dense, margin 0.000, dense +0.000

[1208] Can a landlord evict a tenant for remaining on property after expiration of the [...]
   leg-1   0.000 / 0.000 / 0.000
   leg-2   0.000 / 0.000 / 0.000   -> dense, margin 0.000, dense +0.000

[1134] Can a landlord evict a tenant for nuisance activity? This includes: maintaining, [...]
   leg-1   0.000 / 0.000 / 0.000
   leg-2   0.000 / 0.000 / 0.000   -> dense, margin 0.000, dense +0.000

[2059] Must a landlord accept a tenant's attempt to cure for material breach? In the [...]
   leg-1   0.000 / 0.000 / 0.000
   leg-2   0.000 / 0.000 / 0.000   -> dense, margin 0.000, dense +0.000

[5551] Secondary methods of service are defined as those methods that may be used if the [...]
   leg-1   0.000 / 0.0

### `tied_to_decisive`

Careful: two opposite things share this bucket. A row tied LOW that jumps to 1.0 is a win. A row tied at 1.0 whose dense collapses is a regression with an identical margin. Check the dense delta sign.

In [28]:
eye('tied_to_decisive', n=5)

tied_to_decisive — all_tied -> broke decisively (margin >= 0.4)
7 rows, showing 5
[2190] Must a landlord accept a tenant's attempt to cure for nonpayment of rent? In the [...]
   leg-1   0.189 / 0.189 / 0.189
   leg-2   1.000 / 0.189 / 0.189   -> dense, margin 0.811, dense +0.811

[5336] Secondary methods of service are defined as those methods that may be used if the [...]
   leg-1   1.000 / 1.000 / 1.000
   leg-2   0.150 / 1.000 / 0.189   -> sparse, margin 0.811, dense -0.850

[3104] Is it unlawful to evict a tenant because of the type of income they receive, their [...]
   leg-1   0.087 / 0.087 / 0.087
   leg-2   1.000 / 0.087 / 0.189   -> dense, margin 0.811, dense +0.913

[5331] Secondary methods of service are defined as those methods that may be used if the [...]
   leg-1   1.000 / 1.000 / 1.000
   leg-2   0.189 / 1.000 / 0.189   -> sparse, margin 0.811, dense -0.811

[5334] Secondary methods of service are defined as those methods that may be used if the [...]
   leg-1   1.000 

### `tied_to_thin`

Dense improved and won, narrowly. Sparse did not move, so the movement is real - just small. By the working bar these count.

In [29]:
eye('tied_to_thin', n=5)

tied_to_thin — all_tied -> broke thin, dense UP (0 < margin < 0.1)
36 rows, showing 5
[1675] Can a landlord evict a tenant because they seek to use the property for personal [...]
   leg-1   0.841 / 0.841 / 0.841
   leg-2   0.935 / 0.841 / 0.841   -> dense, margin 0.095, dense +0.095

[1661] Can a landlord evict a tenant for commiting substantial damage to the property? In [...]
   leg-1   0.841 / 0.841 / 0.841
   leg-2   0.933 / 0.841 / 0.841   -> dense, margin 0.093, dense +0.093

[3968] Are eviction cases first heard in county court? In the state of Arkansas
   leg-1   0.050 / 0.050 / 0.050
   leg-2   0.171 / 0.050 / 0.096   -> dense, margin 0.076, dense +0.121

[3971] Are eviction cases first heard in state court? In the state of Arkansas
   leg-1   0.042 / 0.042 / 0.042
   leg-2   0.171 / 0.042 / 0.096   -> dense, margin 0.076, dense +0.130

[7033] In an eviction action, can a tenant rebut/raise the defense that criminal activity [...]
   leg-1   0.079 / 0.079 / 0.079
   leg-2   0

### `tied_backwards`

The tie broke only because dense got worse. Sparse "wins" by standing still. Never read these as the encoder helping.

In [ ]:
eye(case='tied_backwards', n=5)

tied_backwards — all_tied -> broke because dense went DOWN
21 rows, showing 5
[5336] Secondary methods of service are defined as those methods that may be used if the [...]
   leg-1   1.000 / 1.000 / 1.000
   leg-2   0.150 / 1.000 / 0.189   -> sparse, margin 0.811, dense -0.850

[5335] Secondary methods of service are defined as those methods that may be used if the [...]
   leg-1   1.000 / 1.000 / 1.000
   leg-2   0.116 / 1.000 / 0.189   -> sparse, margin 0.811, dense -0.884

[5334] Secondary methods of service are defined as those methods that may be used if the [...]
   leg-1   1.000 / 1.000 / 1.000
   leg-2   0.150 / 1.000 / 0.189   -> sparse, margin 0.811, dense -0.850

[5331] Secondary methods of service are defined as those methods that may be used if the [...]
   leg-1   1.000 / 1.000 / 1.000
   leg-2   0.189 / 1.000 / 0.189   -> sparse, margin 0.811, dense -0.811

[2327] Are fines assessed to landlords for unlawfully evicting a tenant? In the state of [...]
   leg-1   0.884 / 

### `destroyed`

The cost side. These had a retrieved judged doc and now have none - labels lost outright.

In [31]:
eye('destroyed', n=5)

destroyed — had a hit -> lost it entirely
14 rows, showing 5
[7569] Is the term 'Writ of restitution' used to refer to the order from the court to the [...]
   leg-1   0.107 / 0.000 / 0.000
   leg-2   0.000 / 0.000 / 0.000   -> dense, margin 0.000, dense -0.107

[122] Does the law regulating residential evictions specify the type(s) of landlord(s) [...]
   leg-1   1.000 / 0.000 / 0.195
   leg-2   0.000 / 0.000 / 0.000   -> dense, margin 0.000, dense -1.000

[1014] Can a landlord evict a tenant for removal of unit from market? This includes when [...]
   leg-1   0.042 / 0.000 / 0.000
   leg-2   0.000 / 0.000 / 0.000   -> dense, margin 0.000, dense -0.042

[92] With regards to eviction law, does the jurisdiction have separate legal provisions [...]
   leg-1   0.087 / 0.000 / 0.000
   leg-2   0.000 / 0.000 / 0.000   -> dense, margin 0.000, dense -0.087

[7207] In an eviction action, can a tenant rebut/raise the defense that they lawfully [...]
   leg-1   0.821 / 0.000 / 0.057
   leg-2   0

---
## Aggregates per bucket

The same three buckets as counts rather than examples, if you want the totals after eyeballing the cases above. `show()` prints where one bucket's rows went plus their score table.

In [18]:
def show(bucket):
    """Rows drawn from one leg-1 bucket: where they went, and the scores."""
    rows = paired[paired.bucket_1 == bucket]
    if rows.empty:
        print(f'{bucket}: not sampled yet — add it to DRAWS and re-run cells 1-3')
        return None
    moved = rows[rows.bucket_1 != rows.bucket_2]
    print(f'{bucket}: {len(rows)} rows   stayed {len(rows) - len(moved)}   moved {len(moved)}')
    print(rows['bucket_2'].value_counts().to_string())
    cols = ['query_id'] + [c + s for c in ('dense', 'sparse', 'rrf') for s in ('_1', '_2')]
    out = rows.rename(columns={k + s: v + s for k, v in SHORT.items() for s in ('_1', '_2')})
    return (out[cols + ['bucket_2', 'winner_2']]
            .sort_values('dense_2', ascending=False).round(3))


print('helper ready')

helper ready


---
# A. `all_tied` — all three scores the same

Every route already did equally well. There are only two things that can happen:

- the scores stay level → **still tied**, the encoder changed nothing here;
- one route pulls apart from the others → **`routes_differ`**.

**Watch the direction.** If the tie was at the top (all three at 1.0, the judged doc already at rank 1) then dense cannot go *up* — so a tie that breaks there broke because dense got **worse**, and the row's new winner will be sparse. Compare `dense_1` to `dense_2` before reading a broken tie as an improvement.

In [19]:
show('all_tied')

all_tied: 107 rows   stayed 46   moved 61
bucket_2
routes_differ    61
all_tied         46


,query_id,dense_1,dense_2,sparse_1,sparse_2,rrf_1,rrf_2,bucket_2,winner_2
497,2560,1.000,1.000,1.000,1.000,1.000,1.000,all_tied,dense
140,8624,1.000,1.000,1.000,1.000,1.000,1.000,all_tied,dense
307,2190,0.189,1.000,0.189,0.189,0.189,0.189,routes_differ,dense
170,6346,1.000,1.000,1.000,1.000,1.000,1.000,all_tied,dense
417,144,1.000,1.000,1.000,1.000,1.000,1.000,all_tied,dense
...,...,...,...,...,...,...,...,...,...
359,5337,1.000,0.095,1.000,1.000,1.000,1.000,routes_differ,rrf
35,8888,0.189,0.090,0.189,0.189,0.189,0.129,routes_differ,sparse
104,691,0.071,0.071,0.071,0.071,0.071,0.079,routes_differ,rrf
322,1900,0.129,0.000,0.129,0.129,0.129,0.100,routes_differ,sparse


In [20]:
t = paired[paired.bucket_1 == 'all_tied']
if not t.empty:
    delta = t.score_dense_only_2 - t.score_dense_only_1
    print(f'dense went UP   on {int((delta > 1e-9).sum()):>4}')
    print(f'dense went DOWN on {int((delta < -1e-9).sum()):>4}   <-- these are regressions')
    print(f'dense unchanged on {int((delta.abs() <= 1e-9).sum()):>4}')
    print('\nthe ties that broke were at what score level?')
    broke = t[t.bucket_2 != 'all_tied']
    print(broke.assign(tied_at=broke.max_1.round(2),
                       dense=(broke.score_dense_only_2 - broke.score_dense_only_1)
                       .gt(0).map({True: 'dense up', False: 'dense down'}))
          .groupby(['tied_at', 'dense']).size().rename('rows').to_string())

dense went UP   on   39
dense went DOWN on   21   <-- these are regressions
dense unchanged on   47

the ties that broke were at what score level?
tied_at  dense     
0.04     dense up       1
0.05     dense up       2
0.07     dense down     1
0.08     dense up       3
0.09     dense up       3
0.11     dense up       1
0.12     dense up       4
0.13     dense down     1
         dense up       1
0.15     dense down     2
         dense up       1
0.19     dense down     2
         dense up       3
0.79     dense up       1
0.82     dense down     1
0.84     dense up       8
0.88     dense down     2
         dense up      11
1.00     dense down    13


---
# B. `all_zero` — all three scores are 0

No route surfaced anything relevant, even though a judged relevant document exists for the query. Nothing caps these: any score above 0 is an improvement.

**The question:** how many move off zero, and how far. `dense_2 ≥ 0.7` means the judged doc reached **rank 1** (the objective weights a rank-1 hit at 0.7); below that it landed deeper in the top-10.

In [21]:
show('all_zero')

all_zero: 100 rows   stayed 62   moved 38
bucket_2
all_zero         62
routes_differ    38


,query_id,dense_1,dense_2,sparse_1,sparse_2,rrf_1,rrf_2,bucket_2,winner_2
274,7193,0.0,1.0,0.0,0.0,0.0,0.116,routes_differ,dense
401,3441,0.0,1.0,0.0,0.0,0.0,0.189,routes_differ,dense
361,5697,0.0,1.0,0.0,0.0,0.0,0.189,routes_differ,dense
320,4350,0.0,1.0,0.0,0.0,0.0,0.189,routes_differ,dense
445,38,0.0,1.0,0.0,0.0,0.0,0.189,routes_differ,dense
...,...,...,...,...,...,...,...,...,...
146,7924,0.0,0.0,0.0,0.0,0.0,0.000,all_zero,dense
145,2391,0.0,0.0,0.0,0.0,0.0,0.000,all_zero,dense
141,771,0.0,0.0,0.0,0.0,0.0,0.000,all_zero,dense
139,7922,0.0,0.0,0.0,0.0,0.0,0.000,all_zero,dense


In [22]:
z = paired[paired.bucket_1 == 'all_zero']
if not z.empty:
    off = z[z.max_2 > 1e-9]
    print(f'moved off zero : {len(off)}/{len(z)}  ({len(off)/len(z):.0%})')
    if len(off):
        print(f'  judged doc at rank 1 : {int((off.max_2 >= 0.7).sum())}')
        print(f'  deeper in the top-10 : {int((off.max_2 < 0.7).sum())}')
        print('\nwhich route found it:')
        print(off['winner_2'].value_counts().to_string())
    if 'dataset' in z:
        print('\nper lane:')
        print(z.assign(off_zero=z.max_2 > 1e-9).groupby('dataset')['off_zero']
              .agg(rows='size', off_zero='sum').to_string())

moved off zero : 38/100  (38%)
  judged doc at rank 1 : 7
  deeper in the top-10 : 31

which route found it:
winner_2
dense    38

per lane:
                rows  off_zero
dataset                       
crumb-legal-qa   100        38


---
# C. `routes_differ` — the routes already disagree

These rows already say something about dense vs sparse, so the thing that matters is whether they now say something **different**.

**The question:** does the winning route change, and is the flow one-directional? A genuinely better dense encoder should pull rows *toward* dense. Rows flowing the other way are dense losing ground.

In [23]:
show('routes_differ')

routes_differ: 291 rows   stayed 271   moved 20
bucket_2
routes_differ    271
all_zero          14
all_tied           6


,query_id,dense_1,dense_2,sparse_1,sparse_2,rrf_1,rrf_2,bucket_2,winner_2
0,296,0.116,1.0,0.116,0.116,0.129,1.000,routes_differ,dense
437,3157,1.000,1.0,0.058,0.058,0.208,0.195,routes_differ,dense
185,1981,0.972,1.0,0.070,0.070,0.209,0.961,routes_differ,dense
432,8899,0.107,1.0,0.000,0.000,0.000,0.189,routes_differ,dense
174,3839,0.116,1.0,0.000,0.000,0.090,0.189,routes_differ,dense
...,...,...,...,...,...,...,...,...,...
102,92,0.087,0.0,0.000,0.000,0.000,0.000,all_zero,dense
205,711,0.901,0.0,0.050,0.050,0.143,0.044,routes_differ,sparse
204,8380,0.000,0.0,0.116,0.116,0.000,0.095,routes_differ,sparse
199,7762,0.000,0.0,1.000,1.000,0.189,0.189,routes_differ,sparse


In [24]:
d = paired[paired.bucket_1 == 'routes_differ']
if not d.empty:
    print('winning route, leg-1 (rows) x leg-2 (cols):')
    print(pd.crosstab(d['winner_1'], d['winner_2'], margins=True).to_string())
    flips = d[d.winner_1 != d.winner_2]
    print(f'\nwinner changed on {len(flips)}/{len(d)} rows')
    if len(flips):
        to_dense = int((flips.winner_2 == 'dense').sum())
        from_dense = int((flips.winner_1 == 'dense').sum())
        print(f'  toward dense : {to_dense}')
        print(f'  away from dense: {from_dense}')
        print('  -> ' + ('dense gaining ground' if to_dense > from_dense
                         else 'dense losing ground' if from_dense > to_dense
                         else 'no net direction'))

winning route, leg-1 (rows) x leg-2 (cols):
winner_2  dense  rrf  sparse  All
winner_1                         
dense       238    2       2  242
rrf          12    8       2   22
sparse        8   10       9   27
All         258   20      13  291

winner changed on 36/291 rows
  toward dense : 20
  away from dense: 4
  -> dense gaining ground


---
## Reading it

- **`sparse` changed on any row** → stop, the comparison is not controlled.
- **A: ties break with `dense down`** → the encoder is worse on those queries. A broken tie is not automatically a win.
- **B: high off-zero rate** → this is the population where a stronger encoder actually pays, since nothing caps it.
- **C: winner flips away from dense** → the new encoder is losing ground on rows that already had an opinion.

One caveat that survives the simplification: `natural_only=True` by default, so `supplemented` rows (scored against a corpus carrying constructed documents) are excluded — an encoder effect there would be tangled up with the augmentation. `antique` has no natural tied rows at all, so it cannot appear in section A.

To cross-check, set `ENCODER = e5_dense_cfg` and re-run into a different `out_dir`. Two encoders agreeing is much stronger than one.